# module-modules-iter-isinstance-dispatch — ex2: named_modules dispatch — qualified-name to layer-type-label dict

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `module-modules-iter-isinstance-dispatch`. Running the final beacon cell reports progress against the `GAN: model.modules() isinstance dispatch` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: model.modules() isinstance dispatch` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`module-modules-iter-isinstance-dispatch`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "module-modules-iter-isinstance-dispatch"
DD_SUBTOPIC = "GAN: model.modules() isinstance dispatch"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `named_modules()` + dotted-name dispatch

Ex1 counted layers by type — types only, no names. Real training-script instrumentation usually wants the QUALIFIED NAME plus the type:

```python
for qname, m in model.named_modules():
    if isinstance(m, nn.Conv2d):
        report[qname] = 'conv2d'
```

The qname looks like `'features.0'` or `'encoder.block1.bn'`. Dots are module-attribute names; numbers are Sequential indices.

**Empty string for the root module.** `named_modules()` yields `('', model)` first — the root has no qualified name. Most reporting code filters that out (it's the container, not a meaningful submodule).

**Why dotted names over plain types.** Two BatchNorm2d layers in a ResNet block need to be distinguishable in a layer-wise LR schedule, freezing schedule, or weight-decay-exclusion list. The qname is the unique identifier.

### Exercise 2 — named_modules dispatch — qualified-name to layer-type-label dict

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze a model via `model.named_modules()`, dispatching on isinstance to build a `dict[qualified_name, type_label]` that skips the root entry and labels the standard DCGAN layer types.
> Keywords: named-modules, isinstance, dispatch, qualified-name
> ```

**KCs targeted:** `named-modules-iter-qname`, `skip-root-empty-qname`

Implement `ex2_named_layer_report(model)`. Walk every submodule by qualified name, dispatch by type:

1. Build an empty dict `report`.
2. Iterate `for qname, m in model.named_modules()`. SKIP the root entry (`qname == ''`).
3. Dispatch with `if / elif` (mutually exclusive):
   - `nn.Conv2d` → `report[qname] = 'conv2d'`
   - `nn.ConvTranspose2d` → `report[qname] = 'convtranspose2d'`
   - `nn.BatchNorm2d` or `nn.BatchNorm1d` → `report[qname] = 'batchnorm'`
   - `nn.Linear` → `report[qname] = 'linear'`
   - Other → SKIP (do not add to report; the report is type-filtered, not type-complete).
4. Return `report`.

Key differences from ex1:
- ex1 returned a `dict[type_label, count]`; ex2 returns a `dict[qname, type_label]` (no 'other' bucket).
- Root is SKIPPED — `'' -> 'sequential'` would be noise.
- Activations / Flatten / Sequential / ReLU don't appear.

Input: `model` — `nn.Module`.
Output: `dict[str, str]`.

The visualization renders the report as a categorical scatter (x=insertion order, y=type-label, point label = qname).

In [ ]:
def ex2_named_layer_report(model: nn.Module) -> dict:
    report = {}
    for qname, m in model.named_modules():
        if qname == '':
            continue
        if isinstance(m, nn.Conv2d):
            report[qname] = 'conv2d'
        elif isinstance(m, nn.ConvTranspose2d):
            report[qname] = 'convtranspose2d'
        elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
            report[qname] = 'batchnorm'
        elif isinstance(m, nn.Linear):
            report[qname] = 'linear'
    return report


<details><summary>Solution</summary>

```python
def ex2_named_layer_report(model: nn.Module) -> dict:
    report = {}
    for qname, m in model.named_modules():
        if qname == '':
            continue
        if isinstance(m, nn.Conv2d):
            report[qname] = 'conv2d'
        elif isinstance(m, nn.ConvTranspose2d):
            report[qname] = 'convtranspose2d'
        elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
            report[qname] = 'batchnorm'
        elif isinstance(m, nn.Linear):
            report[qname] = 'linear'
    return report
```

**Why filter the root.** `named_modules()` yields the root as `('', model)`. The model itself usually isn't one of the four target types, but even if it were (e.g. a bare `nn.Conv2d`), the empty qname is meaningless for instrumentation. Filter on `qname == ''` rather than relying on the type-check happening to skip it.

**Containers (Sequential, Module subclasses) are naturally filtered.** They're not Conv2d / ConvTranspose2d / BatchNorm / Linear, so they fall through the dispatch without an `else` branch — the report is type-filtered by construction.

**Why `if/elif`, not stacked `if`.** Mutually exclusive — every qname maps to exactly one type-label. Stacked `if` would allow a future subclass that satisfies two type checks to silently overwrite its earlier label.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()